In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
        .master('local[2]') \
        .appName('groupby_join') \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 23:45:35 WARN Utils: Your hostname, BlackBeast, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/08 23:45:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 23:45:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_green = spark.read.parquet("../pq/green/*/*/")

26/03/08 23:45:46 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ./data/pq/green/*/*/.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveD

In [4]:
df_green.registerTempTable('green')

/home/nayya/de-zoomcamp/.venv/lib/python3.13/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [5]:
df_green_rev = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    ROUND(SUM(total_amount),2) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE 
    lpep_pickup_datetime >= '2020-01-01'
GROUP BY
    1,2
""")

In [6]:
df_green_rev \
    .repartition(20) \
    .write.parquet('../report/revenue/green', mode='overwrite')

test = spark.read.parquet("../report/revenue/green/part-00000-ca0e36cb-0fe3-42ac-9077-149a129d17a6-c000.snappy.parquet")

test.count(), len(test.columns)

In [7]:
df_yellow = spark.read.parquet("../pq/yellow/*/*/")
df_yellow.registerTempTable('yellow')
df_yellow_rev = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour,
    PULocationID AS zone,
    ROUND(SUM(total_amount),2) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE 
    tpep_pickup_datetime >= '2020-01-01'
GROUP BY
    1,2
""")
df_yellow_rev \
    .repartition(20) \
    .write.parquet('../report/revenue/yellow/', mode='overwrite')

26/03/08 23:46:05 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ./data/pq/yellow/*/*/.
java.io.FileNotFoundException: File data/pq/yellow/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Resolv

test = spark.read.parquet("../report/revenue/yellow")
test.count(), len(test.columns)

In [17]:
df_green_rev = spark.read.parquet("../report/revenue/green/")
df_yellow_rev = spark.read.parquet("../report/revenue/yellow/")

In [18]:
df_green_rev_tmp = df_green_rev \
                    .withColumnRenamed('amount', 'green_amount') \
                    .withColumnRenamed('number_records', 'green_num_records')

df_yellow_rev_tmp = df_yellow_rev \
                    .withColumnRenamed('amount', 'yellow_amount') \
                    .withColumnRenamed('number_records', 'yellow_num_records')

In [19]:
df_join = df_green_rev_tmp.join(df_yellow_rev_tmp,on=['hour', 'zone'], how='outer')

In [20]:
df_join.show()

+-------------------+----+------------+-----------------+-------------+------------------+
|               hour|zone|green_amount|green_num_records|yellow_amount|yellow_num_records|
+-------------------+----+------------+-----------------+-------------+------------------+
|2020-01-01 00:00:00|   3|        NULL|             NULL|         25.0|                 1|
|2020-01-01 00:00:00|   4|        NULL|             NULL|       1004.3|                57|
|2020-01-01 00:00:00|   7|      769.73|               45|       455.17|                38|
|2020-01-01 00:00:00|  10|        NULL|             NULL|        42.41|                 2|
|2020-01-01 00:00:00|  14|        NULL|             NULL|          8.8|                 1|
|2020-01-01 00:00:00|  18|         7.8|                1|          5.8|                 1|
|2020-01-01 00:00:00|  33|      317.27|               11|       255.56|                 8|
|2020-01-01 00:00:00|  34|        NULL|             NULL|         19.3|                 1|

In [21]:
df_join.write.parquet('../report/revenue/total/', mode='overwrite')

In [22]:
df_join = spark.read.parquet("../report/revenue/total/")

In [23]:
df_join.show()

+-------------------+----+------------+-----------------+-------------+------------------+
|               hour|zone|green_amount|green_num_records|yellow_amount|yellow_num_records|
+-------------------+----+------------+-----------------+-------------+------------------+
|2020-01-01 00:00:00|  12|        NULL|             NULL|        107.0|                 6|
|2020-01-01 00:00:00|  13|        NULL|             NULL|       1214.8|                56|
|2020-01-01 00:00:00|  15|        NULL|             NULL|        34.09|                 1|
|2020-01-01 00:00:00|  17|      195.03|                9|       220.21|                 8|
|2020-01-01 00:00:00|  22|        15.8|                1|         NULL|              NULL|
|2020-01-01 00:00:00|  24|        87.6|                3|       754.95|                45|
|2020-01-01 00:00:00|  25|       531.0|               26|       324.35|                16|
|2020-01-01 00:00:00|  29|        61.3|                1|         NULL|              NULL|

In [24]:
df_zones = spark.read.parquet("./zones.parquet")

In [25]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [29]:
df_results = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [32]:
df_results.drop('LocationID','zone').write.parquet('tmp/revenue-zones')